In [1]:
from azure.storage.blob import ContainerClient
account_name = "dlsaggregatedprodr9"
container_name = "saas-gold-direct-data-access"
sas_token = "sp=rl&st=2025-08-06T11:15:05Z&se=2026-07-31T19:30:05Z&spr=https&sv=2024-11-04&sr=d&sig=022LIiARMF6VCOJ1xaQGH6wv04Fwqh8quWp%2BzX07S7w%3D&sdd=2"
folder_path = "data/studio_id=10720/"
#folder_path = "data/studio_id=[your studio ID]/"


blob_service_url = f"https://{account_name}.blob.core.windows.net/"

container_client = ContainerClient(
    account_url=blob_service_url,
    container_name=container_name,
    credential=sas_token
)



In [2]:
blob_client = container_client.get_blob_client('data/studio_id=10720/export_fact_sales.csv')
blob_client_wishlist = container_client.get_blob_client('data/studio_id=10720/export_fact_visibility_wishlist.csv')
#blob_client_visibility = container_client.get_blob_client('data/studio_id=10720/fact_visibility.csv')

In [3]:
import time
import pandas as pd
from io import BytesIO


t0 = time.perf_counter()
data_rev = blob_client.download_blob().readall()
data_wl = blob_client_wishlist.download_blob().readall()
#data_vis = blob_client_visibility.download_blob().readall()

t1 = time.perf_counter()

In [4]:
df = pd.read_csv(BytesIO(data_rev))
df_wl = pd.read_csv(BytesIO(data_wl))
#df_vis = pd.read_csv(BytesIO(data_vis))

t2 = time.perf_counter()

print(f"Download: {t1 - t0:.3f}s")
print(f"CSV parse: {t2 - t1:.3f}s")
print(f"Total: {t2 - t0:.3f}s")

Download: 89.662s
CSV parse: 7.201s
Total: 96.863s


/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_99397/3286014683.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wl = pd.read_csv(BytesIO(data_wl))


In [5]:
df_wl

,date,unique_sku_id,base_sku_id,human_name,product_id,product_name,portal_platform_region_id,portal,store,country_code,non_owner_visits,non_owner_impressions,adds,deletes,purchases_activations_gifts
0,2010-01-01,1945140-store:10720,1945140,A Memoir Blue - Original Soundtrack,A Memoir Blue - Original Soundtrack:171010:10720,A Memoir Blue - Original Soundtrack,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
1,2010-01-01,1497450-store:10720,1497450,A Memoir Blue,A Memoir Blue:171010:10720,A Memoir Blue,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
2,2010-01-01,1116060-store:10720,1116060,Ashen - Nightstorm Isle,Ashen - Nightstorm Isle DLC:171010:10720,Ashen - Nightstorm Isle DLC,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
3,2010-01-01,1202450-store:10720,1202450,Ashen - Original Soundtrack,Ashen - Original Soundtrack:171010:10720,Ashen - Original Soundtrack,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
4,2010-01-01,649950-store:10720,649950,Ashen,Ashen:171010:10720,Ashen,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333470,2025-09-16,1299460-store:10720,1299460,Wanderstop,Wanderstop:171010:10720,Wanderstop,171010,Steam,Steam,YYY,NaN,NaN,24.0,13.0,1.0
1333471,2025-09-16,702680-store:10720,702680,Wattam,Wattam:171010:10720,Wattam,171010,Steam,Steam,YYY,NaN,NaN,1.0,1.0,0.0
1333472,2025-09-16,501300-store:10720,501300,What Remains of Edith Finch,What Remains of Edith Finch:171010:10720,What Remains of Edith Finch,171010,Steam,Steam,YYY,NaN,NaN,38.0,17.0,2.0
1333473,2025-09-16,1497460-store:10720,1497460,Wheel World,Wheel World:171010:10720,Wheel World,171010,Steam,Steam,YYY,NaN,NaN,10.0,9.0,3.0


In [6]:
df_wl.head().columns

Index(['date', 'unique_sku_id', 'base_sku_id', 'human_name', 'product_id',
       'product_name', 'portal_platform_region_id', 'portal', 'store',
       'country_code', 'non_owner_visits', 'non_owner_impressions', 'adds',
       'deletes', 'purchases_activations_gifts'],
      dtype='object')

In [7]:
df_wl = df_wl.groupby(["date",'product_name'])[['adds','deletes','purchases_activations_gifts','non_owner_visits','non_owner_impressions']].sum().sort_values(by='date',ascending=False).reset_index()

In [8]:
df = df.groupby(["date",'product_name'])[['all_units','units_returned','units_freely_distributed','units_sold_in_retail','gross_revenue', 'gross_returned']].sum().sort_values(by='date',ascending=False).reset_index()

In [9]:
df.columns

Index(['date', 'product_name', 'all_units', 'units_returned',
       'units_freely_distributed', 'units_sold_in_retail', 'gross_revenue',
       'gross_returned'],
      dtype='object')

In [10]:
data = df.merge(df_wl, on=['product_name', 'date'], how='right')

In [11]:
data.groupby("product_name")['units_returned'].sum().sort_values(ascending=False)

product_name
Stray                          257836.0
Outer Wilds                    253598.0
Journey                        126258.0
What Remains of Edith Finch    110263.0
Storyteller                     57342.0
                                 ...   
Forever Ago                         0.0
Morsels                             0.0
Cannula                             0.0
Mixtape                             0.0
The Lost Wild                       0.0
Name: units_returned, Length: 105, dtype: float64

In [12]:
data.groupby("product_name")['all_units'].get_group("Wheel World").sum() - data.groupby("product_name")['units_freely_distributed'].get_group("Wheel World").sum() - data.groupby("product_name")['units_sold_in_retail'].get_group("Wheel World").sum() -data.groupby("product_name")['units_returned'].get_group("Wheel World").sum()

14593.0

In [13]:
data

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned,adds,deletes,purchases_activations_gifts,non_owner_visits,non_owner_impressions
0,2025-09-16,to a T,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0
1,2025-09-16,Gorogoa,NaN,NaN,NaN,NaN,NaN,NaN,7.0,3.0,0.0,0.0,0.0
2,2025-09-16,Lushfoil Photography Sim,NaN,NaN,NaN,NaN,NaN,NaN,18.0,4.0,0.0,0.0,0.0
3,2025-09-16,Lorelei and the Laser Eyes,NaN,NaN,NaN,NaN,NaN,NaN,14.0,3.0,0.0,0.0,0.0
4,2025-09-16,Last Stop,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
444354,2010-01-01,Storyteller,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444355,2010-01-01,Storyteller - Original Soundtrack,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444356,2010-01-01,Stray,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0
444357,2010-01-01,Stray - Original Soundtrack,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0


In [14]:
data['date'] = pd.to_datetime(data['date'])
#data['date_str'] = data['date'].dt.strftime("%Y-%m-%d")

In [15]:
data['units_excl_refunds'] = data['all_units'] - data['units_returned'] - data['units_freely_distributed']- data['units_sold_in_retail']
data['revenue_excl_refunds'] = data['gross_revenue'] - data['gross_returned']

In [16]:
grouped = data.groupby('product_name')

In [17]:
grouped.get_group("Wheel World")['units_returned'].sum()

882.0

In [18]:
rename_dict = {
'date':'Date', 
'product_name':'Product', 
'revenue_excl_refunds':'Revenue (excl. refunds)', 
'units_excl_refunds':'Units (excl. refunds)',
'units_freely_distributed':'Free units', 
    'adds':'Wishlist adds', 
    'non_owner_visits':'Non-owner visits',
       'non_owner_impressions':'Non-owner impressions', 
'units_returned': 'Refunded units',
    'deletes':"wl_deletes",
    'purchases_activations_gifts':"wl_activations",
    
}

In [19]:
data.rename(rename_dict, axis=1, inplace=True)
#data = data.drop_duplicates(subset=['day', 'product'])
#data['day'] = pd.to_datetime(test['day'] )
#data = data.sort_values(by='day')

In [20]:
data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product =='LEGO Voyagers'")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_99397/1039625814.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product =='LEGO Voyagers'")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
691,2025-09-08,LEGO Voyagers,0.0,0.0,19.0,1772.0,5078.0,54936.0,0.0,93.0,0.0


In [21]:
export_df = data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date')

export_df.query("Date=='2025-09-08' & Product =='The Lost Wild'")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_99397/1831163731.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  export_df.query("Date=='2025-09-08' & Product =='The Lost Wild'")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
728,2025-09-08,The Lost Wild,NaN,NaN,NaN,77.0,931.0,16102.0,NaN,78.0,0.0


In [22]:
export_df.tail()

,Date,Product,Revenue (excl. refunds),Units (excl. refunds),Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
28,2025-09-16,Mundaun,NaN,NaN,NaN,7.0,0.0,0.0,NaN,1.0,0.0
27,2025-09-16,Telling Lies,NaN,NaN,NaN,2.0,0.0,0.0,NaN,2.0,0.0
26,2025-09-16,Morsels,NaN,NaN,NaN,3.0,0.0,0.0,NaN,3.0,0.0
36,2025-09-16,The Lost Wild,NaN,NaN,NaN,8.0,0.0,0.0,NaN,10.0,0.0
0,2025-09-16,to a T,NaN,NaN,NaN,1.0,0.0,0.0,NaN,1.0,0.0


In [23]:
export_df.to_csv("DB_Update/API/bulkAPI_Export.csv", index=False)

In [24]:
stop

NameError: name 'stop' is not defined

In [ ]:
stop